# 07 — Lakehouse Monitoring + drift simulation

Closes the MLOps loop. We:

1. **Enable Lakehouse Monitoring** on `shot_predictions` as an `InferenceLog`
   monitor (label + prediction + timestamp), sliced by `shot_type` and `strength_state`.
2. **Simulate drift** — generate a fresh batch of shots with a *shifted* shot-mix
   distribution (think: a new opposing team that takes more low-angle shots
   from the perimeter), score them with `@champion`, and append to the table.
3. **Refresh the monitor** and show the metric tables that just appeared.

In a real demo you'd let the monitor pick up the drift on its scheduled run —
here we force a refresh so we can see it in the same session.

In [1]:
import os, time, math
import numpy as np
import pandas as pd
import mlflow
from datetime import datetime, timezone, timedelta
from dotenv import load_dotenv
load_dotenv()

try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "hockey_xg_mlflow")
MODEL_NAME = f"{UC_CATALOG}.{UC_SCHEMA}.xg_model"
PRED_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.shot_predictions"

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
print(f"Predictions table: {PRED_TABLE}")

/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:141: UserWarning: Could not enable SetAllowOversizeProtos, please check the protobuf version.
  warnings.warn("Could not enable SetAllowOversizeProtos, please check the protobuf version.")


Predictions table: alexander_booth.hockey_xg_mlflow.shot_predictions


## Generate a drifted batch

We change the underlying shot-mix distribution — more long-distance perimeter
shots, fewer rebounds, slightly more PK time. Then we score it with the same
`@champion` model. The model's outputs should still be reasonable, but the
**feature distribution** is now different from the training set — which is
exactly the kind of drift the monitor flags.

In [2]:
rng = np.random.default_rng(seed=7)
N_DRIFT = 8_000

# Shifted distance distribution (further out)
x = 89 - np.clip(rng.gamma(shape=2.5, scale=18.0, size=N_DRIFT), 1, 64)
y = rng.normal(loc=0.0, scale=22.0, size=N_DRIFT)
y = np.clip(y, -42.0, 42.0)
dx = 89.0 - x; dy = y
distance_ft = np.sqrt(dx**2 + dy**2)
angle_deg = np.degrees(np.arctan2(np.abs(dy), np.maximum(dx, 0.5)))

# Shifted shot-type mix — more slap, less wrist
shot_types = np.array(["wrist", "snap", "slap", "backhand", "tip", "wrap"])
shot_p     = np.array([0.30,    0.18,   0.30,   0.10,       0.08,  0.04])
shot_type  = rng.choice(shot_types, size=N_DRIFT, p=shot_p)

# Shifted strength-state mix — much more PK
strength_states = np.array(["5v5", "5v4_pp", "4v5_pk", "4v4", "3v3_ot", "6v5_en", "5v6"])
strength_p      = np.array([0.65,   0.08,    0.18,     0.04,   0.01,     0.02,     0.02])
strength_state  = rng.choice(strength_states, size=N_DRIFT, p=strength_p)

period         = rng.choice([1, 2, 3, 4], size=N_DRIFT, p=[0.32, 0.33, 0.32, 0.03])
rebound        = rng.random(N_DRIFT) < 0.05   # fewer rebounds
rush           = rng.random(N_DRIFT) < 0.10
prev_event_sec = np.where(rebound, rng.uniform(0.5, 3.0, N_DRIFT), rng.exponential(scale=22.0, size=N_DRIFT))
shot_id    = np.arange(900_001, 900_001 + N_DRIFT)
game_id    = rng.integers(low=2024030001, high=2024030100, size=N_DRIFT)
team_id    = rng.integers(low=1, high=33, size=N_DRIFT)
opp_team_id = ((team_id - 1 + rng.integers(1, 32, size=N_DRIFT)) % 32) + 1
player_id  = (team_id - 1) * 25 + rng.integers(1, 26, size=N_DRIFT)
time_in_period_sec = rng.integers(low=0, high=1200, size=N_DRIFT)

# Generate "true" goal under the original ground truth so we still have a label
def sigmoid(z): return 1.0/(1.0+np.exp(-z))
shot_type_eff = pd.Series(shot_type).map({
    "wrist": 0.0, "snap": 0.10, "slap": -0.10, "backhand": -0.25, "tip": 0.55, "wrap": -0.05
}).to_numpy()
strength_eff = pd.Series(strength_state).map({
    "5v5": 0.0, "5v4_pp": 0.35, "4v5_pk": -0.30, "4v4": 0.10,
    "3v3_ot": 0.25, "6v5_en": 2.50, "5v6": -0.40
}).to_numpy()
logit = (-1.5 + -0.055*distance_ft + -0.022*angle_deg + 0.00006*distance_ft**2
         + shot_type_eff + strength_eff + 0.75*rebound + 0.40*rush
         + 0.10*(period == 3).astype(float) + rng.normal(0, 0.20, N_DRIFT))
goal = (rng.random(N_DRIFT) < sigmoid(logit)).astype(int)

pdf = pd.DataFrame({
    "shot_id": shot_id, "game_id": game_id, "period": period.astype(int),
    "time_in_period_sec": time_in_period_sec.astype(int),
    "team_id": team_id.astype(int), "opp_team_id": opp_team_id.astype(int),
    "player_id": player_id.astype(int),
    "shot_type": shot_type, "strength_state": strength_state, "goal": goal.astype(int),
    "distance_ft": distance_ft.round(2), "angle_deg": angle_deg.round(2),
    "distance_sq": (distance_ft**2).round(2), "angle_sq": (angle_deg**2).round(2),
    "is_high_danger": ((distance_ft <= 20) & (angle_deg <= 30)).astype(int),
    "rebound": rebound.astype(int), "rush": rush.astype(int),
    "rebound_dist": (rebound*distance_ft).round(2), "rush_dist": (rush*distance_ft).round(2),
    "prev_event_sec": prev_event_sec.round(2),
})
for st in ["wrist", "snap", "slap", "backhand", "tip", "wrap"]:
    pdf[f"st_{st}"] = (pdf["shot_type"] == st).astype(int)
for s in ["5v5", "5v4_pp", "4v5_pk", "4v4", "3v3_ot", "6v5_en", "5v6"]:
    pdf[f"str_{s}"] = (pdf["strength_state"] == s).astype(int)

print(f"Drift batch ready: {len(pdf):,} rows  goal rate: {pdf['goal'].mean()*100:.2f}%")

Drift batch ready: 8,000 rows  goal rate: 2.46%


In [3]:
# Score the drift batch with @champion (same alias-based load — no version change)
import mlflow.pyfunc
from mlflow.tracking import MlflowClient

FEATURE_COLS = [
    "distance_ft", "angle_deg", "distance_sq", "angle_sq",
    "is_high_danger", "rebound", "rush", "rebound_dist", "rush_dist",
    "period", "time_in_period_sec", "prev_event_sec",
    "st_wrist", "st_snap", "st_slap", "st_backhand", "st_tip", "st_wrap",
    "str_5v5", "str_5v4_pp", "str_4v5_pk", "str_4v4", "str_3v3_ot", "str_6v5_en", "str_5v6",
]

model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
client = MlflowClient()
mv = client.get_model_version_by_alias(MODEL_NAME, "champion")
proba = model.predict(pdf[FEATURE_COLS].astype("float64"))
if hasattr(proba, "values"): proba = proba.values.reshape(-1)
pdf["xg"] = proba
pdf["prediction"]    = (pdf["xg"] >= 0.5).astype(int)
pdf["model_version"] = int(mv.version)
pdf["model_uri"]     = f"models:/{MODEL_NAME}/{mv.version}"
pdf["scored_at_utc"] = datetime.now(timezone.utc)
pdf["batch_id"]      = "drift_simulation"
print(f"Avg xG on drift batch: {pdf['xg'].mean():.4f}  (vs ~{pdf['goal'].mean():.4f} actual goal rate)")

/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Avg xG on drift batch: 0.0480  (vs ~0.0246 actual goal rate)


In [4]:
# Append to predictions table — match target schema exactly to avoid Delta merge issues
target_schema = spark.table(PRED_TABLE).schema
target_cols   = [f.name for f in target_schema]
sdf = spark.createDataFrame(pdf[target_cols])
# Cast each column to the target type
from pyspark.sql import functions as F
for f in target_schema:
    sdf = sdf.withColumn(f.name, F.col(f.name).cast(f.dataType))
sdf.write.mode("append").saveAsTable(PRED_TABLE)
print(f"Appended {sdf.count():,} drift rows. Predictions table now has:")
spark.sql(f"SELECT batch_id, COUNT(*) AS rows, AVG(xg) AS avg_xg, AVG(goal) AS actual_rate FROM {PRED_TABLE} GROUP BY batch_id").show(truncate=False)

Appended 8,000 drift rows. Predictions table now has:


+----------------+-----+--------------------+-----------+
|batch_id        |rows |avg_xg              |actual_rate|
+----------------+-----+--------------------+-----------+
|initial_backfill|50000|0.09378523888174324 |0.09416    |
|drift_simulation|8000 |0.047956323586958886|0.024625   |
+----------------+-----+--------------------+-----------+



## Create the Lakehouse Monitor

`InferenceLog` monitor type — built for ML inference tables. It computes
drift on features + on prediction distribution, and quality metrics (since we
have a label column) on each refresh.

In [5]:
from databricks.sdk.service.catalog import (
    MonitorInferenceLog, MonitorInferenceLogProblemType, MonitorInfoStatus
)

ASSETS_DIR = f"/Users/{w.current_user.me().user_name}/lakehouse_monitor_assets/hockey_xg"

inference_log = MonitorInferenceLog(
    timestamp_col="scored_at_utc",
    granularities=["1 day"],
    model_id_col="model_version",
    prediction_col="prediction",
    label_col="goal",
    problem_type=MonitorInferenceLogProblemType.PROBLEM_TYPE_CLASSIFICATION,
)

# Idempotent create-or-update
existing = None
try:
    existing = w.quality_monitors.get(table_name=PRED_TABLE)
except Exception:
    pass

if existing:
    print(f"Monitor exists — updating")
    w.quality_monitors.update(
        table_name=PRED_TABLE,
        inference_log=inference_log,
        output_schema_name=f"{UC_CATALOG}.{UC_SCHEMA}",
        assets_dir=ASSETS_DIR,
        slicing_exprs=["shot_type", "strength_state"],
    )
else:
    print("Creating monitor")
    w.quality_monitors.create(
        table_name=PRED_TABLE,
        inference_log=inference_log,
        output_schema_name=f"{UC_CATALOG}.{UC_SCHEMA}",
        assets_dir=ASSETS_DIR,
        slicing_exprs=["shot_type", "strength_state"],
    )

print("Monitor configured. Waiting for ACTIVE status...")
for i in range(30):
    info = w.quality_monitors.get(table_name=PRED_TABLE)
    print(f"  [{i:02d}] status={info.status}")
    if info.status == MonitorInfoStatus.MONITOR_STATUS_ACTIVE:
        break
    time.sleep(10)

Monitor exists — updating


TypeError: QualityMonitorsAPI.update() got an unexpected keyword argument 'assets_dir'

In [6]:
# Trigger a refresh so metrics tables get populated now
refresh = w.quality_monitors.run_refresh(table_name=PRED_TABLE)
print(f"Refresh started: {refresh.refresh_id}  state={refresh.state}")
print("First-time refresh can take 10-20 minutes on Databricks. Polling...")
final_state = None
for i in range(80):
    r = w.quality_monitors.get_refresh(table_name=PRED_TABLE, refresh_id=str(refresh.refresh_id))
    final_state = r.state
    print(f"  [{i:02d}] state={r.state}")
    if str(r.state).endswith("SUCCESS") or str(r.state).endswith("FAILED") or str(r.state).endswith("CANCELED"):
        break
    time.sleep(20)
if not str(final_state).endswith("SUCCESS"):
    print("\n⚠️  Refresh did not finish in time. Metric tables may not be populated yet.")
    print("   For the live demo: come back to the dashboard after the refresh completes.")
    print("   You can also re-poll via: w.quality_monitors.get_refresh(table_name=PRED_TABLE, refresh_id='%s')" % refresh.refresh_id)

Refresh started: 875526519281802  state=MonitorRefreshInfoState.RUNNING
First-time refresh can take 10-20 minutes on Databricks. Polling...


  [00] state=MonitorRefreshInfoState.RUNNING


  [01] state=MonitorRefreshInfoState.RUNNING


  [02] state=MonitorRefreshInfoState.RUNNING


  [03] state=MonitorRefreshInfoState.RUNNING


...

  [29] state=MonitorRefreshInfoState.RUNNING


  [30] state=MonitorRefreshInfoState.RUNNING


  [31] state=MonitorRefreshInfoState.RUNNING


  [32] state=MonitorRefreshInfoState.RUNNING


  [33] state=MonitorRefreshInfoState.SUCCESS


## Inspect the metrics tables

The monitor wrote two tables next to the source: `{prefix}_profile_metrics` and
`{prefix}_drift_metrics`. The drift table compares each window to the baseline
(if one is set) and across consecutive windows.

In [7]:
info = w.quality_monitors.get(table_name=PRED_TABLE)
print(f"Profile metrics table: {info.profile_metrics_table_name}")
print(f"Drift metrics table:   {info.drift_metrics_table_name}")
if info.dashboard_id:
    print(f"Dashboard: https://e2-demo-field-eng.cloud.databricks.com/sql/dashboards/{info.dashboard_id}")
else:
    print("Dashboard: (auto-creating, will appear in the monitor UI once the first refresh succeeds)")

Profile metrics table: alexander_booth.hockey_xg_mlflow.shot_predictions_profile_metrics
Drift metrics table:   alexander_booth.hockey_xg_mlflow.shot_predictions_drift_metrics
Dashboard: https://e2-demo-field-eng.cloud.databricks.com/sql/dashboards/01f15b10810c1e84b195b347490ed4ab


In [8]:
# Quick look at drift signals — only valid once the refresh has finished
try:
    spark.sql(f'''
      SELECT window, column_name, drift_type, js_distance, ks_test
      FROM {info.drift_metrics_table_name}
      WHERE column_name IN ('xg', 'prediction', 'distance_ft', 'angle_deg', 'shot_type', 'strength_state')
      ORDER BY window.start DESC, column_name
      LIMIT 50
    ''').show(truncate=False)
except Exception as e:
    print(f"Drift metrics not available yet — refresh still running. Try again in a few minutes.")
    print(f"   Table: {info.drift_metrics_table_name}")

+------------------------------------------+-----------+-----------+-----------+------------------------------+
|window                                    |column_name|drift_type |js_distance|ks_test                       |
+------------------------------------------+-----------+-----------+-----------+------------------------------+
|{2026-05-29 00:00:00, 2026-05-30 00:00:00}|angle_deg  |CONSECUTIVE|NULL       |{0.108, 5.492036122008295E-47}|
|{2026-05-29 00:00:00, 2026-05-30 00:00:00}|angle_deg  |CONSECUTIVE|NULL       |{0.1, 2.5059186778149335E-5}  |
|{2026-05-29 00:00:00, 2026-05-30 00:00:00}|angle_deg  |CONSECUTIVE|NULL       |{0.121, 9.557549405111534E-10}|
|{2026-05-29 00:00:00, 2026-05-30 00:00:00}|angle_deg  |CONSECUTIVE|NULL       |{0.151, 0.0406716096391492}   |
|{2026-05-29 00:00:00, 2026-05-30 00:00:00}|angle_deg  |CONSECUTIVE|NULL       |{0.151, 0.0406716096391492}   |
|{2026-05-29 00:00:00, 2026-05-30 00:00:00}|angle_deg  |CONSECUTIVE|NULL       |{0.133, 0.01148019147777

**That's the loop.** Open the auto-generated dashboard from the monitor UI
to walk the audience through the drift visualizations.

You can also wire a **DBSQL alert** on the drift metrics table — e.g., page on
`js_distance > 0.2` for the `shot_type` column — but that's outside the scope
of a one-hour demo.